In [1]:
import numpy as np

In [2]:
ratings = {
    "David Smith": {
        "Vertigo": 4,
        "Scarface": 4.5,
        "Raging Bull": 3.0,
        "Goodfellas": 4.5,
        "The Apartment": 1.0
    },
    "John Carson": {
        "Vertigo": 4.5,
        "Scarface": 5.0,
        "Raging Bull": 3.5,
        "Goodfellas": 5.0,
        "The Apartment": 1.5,
        "Taxi Driver": 4.0
    },
    "Michelle Lee": {
        "Vertigo": 2.0,
        "Scarface": 2.5,
        "Raging Bull": 5.0,
        "Goodfellas": 3.0,
        "The Apartment": 4.5,
        "The Godfather": 4.0
    },
    "Sarah Parker": {
        "Vertigo": 4.0,
        "Scarface": 4.0,
        "Raging Bull": 3.0,
        "Goodfellas": 4.0,
        "The Apartment": 1.0,
        "Taxi Driver": 4.5
    },
    "Tom White": {
        "Vertigo": 1.5,
        "Scarface": 2.0,
        "Raging Bull": 4.5,
        "Goodfellas": 2.5,
        "The Apartment": 5.0,
        "The Godfather": 4.5
    }
}

In [3]:
def pearson_score(dataset, user1, user2):
    if user1 not in dataset:
        raise TypeError("User " + user1 + " not present in dataset")

    if user2 not in dataset:
        raise TypeError("User " + user2 + " not present in dataset")

    common_movies = {}

    for item in dataset[user1]:
        if item in dataset[user2]:
            common_movies[item] = 1

    num_ratings = len(common_movies)

    if num_ratings == 0:
        return 0

    user1_sum = np.sum([dataset[user1][item] for item in common_movies])
    user2_sum = np.sum([dataset[user2][item] for item in common_movies])

    user1_squared_sum = np.sum([np.square(dataset[user1][item]) for item in common_movies])
    user2_squared_sum = np.sum([np.square(dataset[user2][item]) for item in common_movies])

    product_sum = np.sum([
        dataset[user1][item] * dataset[user2][item]
        for item in common_movies
    ])

    numerator = product_sum - (user1_sum * user2_sum / num_ratings)

    denominator = np.sqrt(
        (user1_squared_sum - np.square(user1_sum) / num_ratings) *
        (user2_squared_sum - np.square(user2_sum) / num_ratings)
    )

    if denominator == 0:
        return 0

    return numerator / denominator

In [4]:
def generate_recommendations(dataset, user):
    if user not in dataset:
        raise TypeError("User " + user + " not present in dataset")

    total_scores = {}
    similarity_sums = {}

    for other_user in dataset:
        if other_user == user:
            continue

        similarity_score = pearson_score(dataset, user, other_user)

        if similarity_score <= 0:
            continue

        for item in dataset[other_user]:
            if item not in dataset[user] or dataset[user][item] == 0:
                total_scores.setdefault(item, 0)
                total_scores[item] += dataset[other_user][item] * similarity_score

                similarity_sums.setdefault(item, 0)
                similarity_sums[item] += similarity_score

    rankings = [
        (total_scores[item] / similarity_sums[item], item)
        for item in total_scores
    ]

    rankings.sort(reverse=True)

    return rankings

In [5]:
user = "David Smith"

recommendations = generate_recommendations(ratings, user)

print("Movie recommendations for", user)
print()

for score, movie in recommendations:
    print(movie, ":", score)

Movie recommendations for David Smith

Taxi Driver : 4.2485031964319315
